<a href="https://colab.research.google.com/github/vinothkumard2006-ai/Practice-in-colab/blob/main/Political_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install nbformat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.listdir('/content/drive/MyDrive')

In [ ]:
Data = pd.read_csv('/content/drive/MyDrive/final_combined_enriched_v4.csv')

In [ ]:
Data.head()

In [ ]:
Data.tail()

In [ ]:
Data.shape

In [ ]:
Data["type"].value_counts()

## Check missing values

In [ ]:
Data.isna().sum()

## Very important (Text cleaning)

In [ ]:
## WE clean
#1. lowercase
#2. remove URls
#3. remove mentions & hastags
#4. remove punctuation
#5. remove stopwords

In [ ]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words=set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'https\S+', '', text)   ## remove urls
    text = re.sub(r'@\w+', '', text)  ## remove mentions
    text = re.sub(r'#\w+', '', text)  ## remove hastags
    text = re.sub(r'[^a-z\s]', '', text)  ## remove punctuations

    text = " ".join([word for word in text.split() if word not in stop_words])

    return text

Data['clean_tweet']=Data['tweet'].apply(clean_text)


## step 4   Train test split

In [ ]:
from sklearn.model_selection import train_test_split

X=Data['clean_tweet']
y=Data['type']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

## step 5 Convert text to numbers(TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
             max_features=5000,
             ngram_range=(1,2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)


## step 6 Train Multiple Models

In [ ]:
## 1. Logistic Regression

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train_vec, y_train)

In [ ]:
## 2. Navie Bayes

from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)

In [ ]:
## 3. Support Vector Machine(SVM)

from sklearn.svm import LinearSVC
svm = LinearSVC()
svm.fit(X_train_vec, y_train)

## Step 7 Evaluate Models

In [ ]:
from sklearn.metrics import accuracy_score,classification_report

def evaluate(model):
    y_pred = model.predict(X_test_vec)
    print("Accuracy:",accuracy_score(y_test,y_pred))


    print(classification_report(y_test,y_pred))
print("Logistic Regression")
evaluate(lr)

print("Navie Bayes")
evaluate(nb)

print("Support Vector Machine")
evaluate(svm)

##  Select Highest accuray and f1 score

##  Step 8 Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {'C':[0.1,1,10]}

grid = GridSearchCV(
    LogisticRegression(),
    params,
    cv=5,
    scoring="f1")

grid.fit(X_train_vec,y_train)
grid.best_params_

In [ ]:
## Train best model

best_model = grid.best_estimator_

## Step 9 Save Model

In [ ]:
import joblib
joblib.dump(best_model,"political_model.pkl")
joblib.dump(tfidf,"tfidf.pkl")

## Step 10  Predict New sentence

In [ ]:
model = joblib.load("political_model.pkl")
vectorizer = joblib.load("tfidf.pkl")

def predict_sentence(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    result = model.predict(vec)[0]

    return "Political"  if result == 1 else "Non-Political"

predict_sentence("The decision sparked debates across universities and workplaces") ## model is successfully  predicted !!!!!

# Deep Learning

### Train test split

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
train_texts, test_texts, train_labels, test_labels=train_test_split(
    Data['tweet'].tolist(),
    Data['type'].tolist(),
    test_size=0.2,
    random_state=42)

## Load BERT Tokenizer

In [ ]:
import torch

from transformers import BertForSequenceClassification
from transformers import Trainer,TrainingArguments

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("distilbert-base-uncased")

## Tokenization

In [ ]:

train_encodings = tokenizer(
    train_texts,truncation=True,
    padding=True,
    max_length=128)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128)

## Create Torch Dataset

In [ ]:
class TweetDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx])
for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = TweetDataset(train_encodings, train_labels)
test_dataset = TweetDataset(test_encodings, test_labels)

# Load Pretrained BERT Model

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2)

# Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,fp16=True,
    save_total_limit=2,
    report_to="none")

# Trainer

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics)

# Train model

In [ ]:
trainer.train()   ##take time 15-40minutes (GPU faster)

# Evaluate Model

In [ ]:
trainer.evaluate()

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Now when you evaluate:
trainer.evaluate()

In [ ]:
import json
with open("/content/drive/MyDrive/Colab Notebooks/Political classifier.ipynb","r") as f:
    nb = json.load(f)
if "widgets" in nb.get("metadata", {}):
    nb["metadata"].pop("widgets")
with open("Political_classifier_clean.ipynb","w") as f:
    json.dump(nb,f)